In [ ]:
# Ass8
# UCS420 – Cognitive Computing | Lab Assignment 10
# NLP using Python-II: Feature extraction from text, sentiment analysis and text generation
# Setup

# Install required libraries (run once; skip if already installed)
!pip install -q nltk textblob wordcloud scikit-learn tensorflow matplotlib

In [ ]:
import re
import string
import numpy as np
import matplotlib.pyplot as plt

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'vader_lexicon', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words('english'))

In [ ]:
# Q1. Text preprocessing on a custom paragraph
# Topic: Punjabi street food

paragraph = ("""Punjabi street food is a celebration of bold flavours, generous portions and warm hospitality. 
On a cold winter evening in Patiala, nothing beats a plate of crispy aloo tikki topped with tangy chutneys and spicy chole. 
Every street vendor guards a secret masala blend that makes their food taste different from the stall next door. 
Amritsari kulchas, stuffed with potatoes and baked in a clay tandoor, are served with a big dollop of butter! 
For dessert, a glass of thick lassi or a bowl of hot gulab jamun completes the meal perfectly. 
Eating street food in Punjab is not just about taste; it is about sharing stories, laughter and food with friends.""")
print(paragraph)

In [ ]:
# 1. Lowercase and remove punctuation using re

lower_text = paragraph.lower()
clean_text = re.sub(r'[^\w\s]', '', lower_text)     # remove everything that is not a word char or whitespace
clean_text = re.sub(r'\s+', ' ', clean_text).strip() # collapse extra spaces/newlines
print(clean_text)

In [ ]:
# 2. Tokenize into sentences and words

sentences = sent_tokenize(paragraph)        # sentence tokenization needs punctuation, so use the original text
print("Number of sentences:", len(sentences))
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")

words = word_tokenize(clean_text)
print("\nNumber of word tokens:", len(words))
print(words)

In [ ]:
# 3. split() vs word_tokenize()

sample = sentences[3]   # the kulcha sentence has commas and an exclamation mark
split_tokens = sample.split()
nltk_tokens = word_tokenize(sample)

print("Sentence       :", sample)
print("\nsplit()        :", split_tokens, "->", len(split_tokens), "tokens")
print("word_tokenize():", nltk_tokens, "->", len(nltk_tokens), "tokens")

# Same comparison on the whole raw paragraph
print("\nWhole paragraph -> split():", len(paragraph.split()), "| word_tokenize():", len(word_tokenize(paragraph)))
print("Tokens only produced by split():", sorted(set(paragraph.split()) - set(word_tokenize(paragraph))))

In [ ]:
# Observation:
# - str.split() only breaks on whitespace, so punctuation stays glued to words ("kulchas,", "butter!", "taste;"). These would be counted as different words from "kulchas" or "butter".
# - NLTK's word_tokenize() is language-aware: it separates punctuation into its own tokens (",", "!", ";") and also handles contractions (e.g. "don't" → "do", "n't"). That is why it produces more tokens.
# - For NLP tasks word_tokenize() is the better choice; split() is only fine when the text has already been cleaned.

In [ ]:
# 4. Remove stopwords

filtered_words = [w for w in words if w not in stop_words]
print("Before:", len(words), "tokens | After:", len(filtered_words), "tokens")
print(filtered_words)

In [ ]:
# 5. Word frequency distribution (excluding stopwords)

fdist = FreqDist(filtered_words)
print("Top 10 most common words:")
for word, freq in fdist.most_common(10):
    print(f"{word:12s} {freq}")

top = fdist.most_common(15)
plt.figure(figsize=(10, 4))
plt.bar([w for w, _ in top], [c for _, c in top], color='teal')
plt.xticks(rotation=45, ha='right')
plt.title('Word Frequency Distribution (stopwords removed)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Q2. Stemming vs Lemmatization (same paragraph)
# 1. Extract alphabet-only words using re.findall()

alpha_words = re.findall(r'\b[a-zA-Z]+\b', paragraph.lower())
print(len(alpha_words), "words")
print(alpha_words)

In [ ]:
# 2. Remove stopwords

q2_words = [w for w in alpha_words if w not in stop_words]
print(len(q2_words), "words after stopword removal")
print(q2_words)

In [ ]:
# 3. Stemming with PorterStemmer

stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in q2_words]
print(stemmed)

In [ ]:
# 4. Lemmatization with WordNetLemmatizer

from nltk import pos_tag
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

# Default lemmatization (treats every word as a noun)
lemmatized_default = [lemmatizer.lemmatize(w) for w in q2_words]

# POS-aware lemmatization (more accurate)
def to_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

try:
    tagged = pos_tag(q2_words)
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    tagged = pos_tag(q2_words)

lemmatized = [lemmatizer.lemmatize(w, to_wordnet_pos(t)) for w, t in tagged]
print("Noun-only lemmas:", lemmatized_default)
print("\nPOS-aware lemmas:", lemmatized)

In [ ]:
# 5. Compare stemmed vs lemmatized output

print(f"{'Original':15s}{'Stemmed':15s}{'Lemmatized':15s}")
print('-' * 45)
for o, s, l in zip(q2_words, stemmed, lemmatized):
    flag = '  <-- differs' if s != l else ''
    print(f"{o:15s}{s:15s}{l:15s}{flag}")

In [ ]:
# Comparison and when to prefer which:
#
#  | Stemming (Porter) | Lemmatization (WordNet)
# How it works | Chops suffixes using fixed rules | Looks up the dictionary form using vocabulary + POS
# Output | Often not a real word (celebration → celebr, hospitality → hospit, generous → gener) | Always a valid word (celebration, served → serve, stories → story)
# Speed | Very fast | Slower (dictionary lookup, needs POS tag for best results)
# Accuracy | Lower – can over-stem (generous/general → gener) | Higher – context aware
#
# - Prefer stemming when speed and recall matter more than readability: search engines, information retrieval, large-scale document clustering, or quick BoW/TF-IDF features where you only need words like *serve/served/serving* to collapse together.
# - Prefer lemmatization when meaning and readable output matter: chatbots, question answering, sentiment analysis, text generation, or anything a human will read, since it returns real dictionary words and respects part of speech.

In [ ]:
# Q3. Bag of Words and TF-IDF
# Three short texts of my own (product reviews):

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

texts = [
    "The new smartphone has an amazing camera and the battery lasts all day, but the phone heats up while gaming.",
    "This laptop is lightweight and fast, the keyboard feels great and the battery easily lasts a full college day.",
    "The wireless earbuds have deep bass and clear calls, but the case feels cheap and the earbuds keep disconnecting."
]
labels = ['Text 1 (Phone)', 'Text 2 (Laptop)', 'Text 3 (Earbuds)']
for l, t in zip(labels, texts):
    print(f"{l}: {t}")

In [ ]:
# 1. Bag of Words with CountVectorizer

cv = CountVectorizer(stop_words='english')
bow = cv.fit_transform(texts)

bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=labels)
print("Vocabulary size:", len(cv.vocabulary_))
bow_df

In [ ]:
# 2. TF-IDF with TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(texts)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=labels).round(3)
tfidf_df

In [ ]:
# 3. Top 3 keywords per text

for label in labels:
    top3 = tfidf_df.loc[label].sort_values(ascending=False).head(3)
    print(f"{label}:")
    for word, score in top3.items():
        print(f"   {word:12s} {score:.3f}")
    print()

In [ ]:
# Interpretation:
# - TF-IDF gives high scores to words that are frequent in one text but rare across the other texts.
# - Words like battery, lasts, day and feels appear in two of the three reviews, so their IDF is lower and they get pushed down, even though they are meaningful.
# - Text 3 is dominated by earbuds because it appears twice there and nowhere else — exactly the word that identifies the topic.
# - In Text 1 and Text 2, every distinctive word appears only once, so many words tie for the top score (e.g. *amazing, camera, smartphone, gaming* for the phone; *keyboard, fast, college, laptop* for the laptop). Which three are printed among the tied words is arbitrary, but all of them are words unique to that review, which is exactly what TF-IDF is designed to surface.
# - Compared with plain Bag of Words (which only counts), TF-IDF does a better job of picking out the words that make each review *different* from the others.

In [ ]:
# Q4. Jaccard vs Cosine similarity
# Two short texts describing different technologies: Artificial Intelligence vs Blockchain.

text_ai = """Artificial intelligence enables machines to learn patterns from large amounts of data.
AI systems use algorithms such as neural networks to make predictions and decisions.
It powers voice assistants, recommendation engines and self-driving cars.
The quality of an AI model depends heavily on the data it is trained on.
AI is transforming healthcare, finance and education."""

text_bc = """Blockchain is a decentralized digital ledger that records transactions across many computers.
Each block stores data and is linked to the previous block using cryptography.
Because the data cannot be changed easily, blockchain provides transparency and security.
It is the technology behind cryptocurrencies like Bitcoin.
Blockchain is also transforming finance, supply chains and healthcare records."""

In [ ]:
# 1. Preprocess and tokenize

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    return [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]

tokens_ai = preprocess(text_ai)
tokens_bc = preprocess(text_bc)
print("AI tokens        :", tokens_ai)
print("\nBlockchain tokens:", tokens_bc)

In [ ]:
# 2a. Jaccard similarity (sets)

set_ai, set_bc = set(tokens_ai), set(tokens_bc)
intersection = set_ai & set_bc
union = set_ai | set_bc
jaccard = len(intersection) / len(union)

print("Common words :", sorted(intersection))
print("|A ∩ B| =", len(intersection), " |A ∪ B| =", len(union))
print(f"Jaccard similarity = {jaccard:.4f}")

In [ ]:
# 2b. Cosine similarity (TF-IDF)

from sklearn.metrics.pairwise import cosine_similarity

docs = [' '.join(tokens_ai), ' '.join(tokens_bc)]
vec = TfidfVectorizer()
X = vec.fit_transform(docs)
cosine = cosine_similarity(X[0], X[1])[0][0]
print(f"Cosine similarity (TF-IDF) = {cosine:.4f}")

In [ ]:
# 2c. Which metric gives better insight?

print(f"Jaccard: {jaccard:.4f}   Cosine (TF-IDF): {cosine:.4f}")

In [ ]:
# Analysis:
# - Both scores are low, which is correct: AI and Blockchain are different technologies. The overlap comes from generic words like *data, finance, healthcare, transforming*.
# - Jaccard only checks whether a word is present or not. Every shared word counts equally, and it ignores how often a word occurs. Repeating *data* three times or once makes no difference.
# - Cosine similarity with TF-IDF uses weights: words that appear in both documents get a low IDF (so shared generic words matter less), and frequency within a document is taken into account. It is also normalized for document length.
# - In this case cosine similarity gives better insight because it reflects *how important* the shared words are rather than just counting them. Jaccard is simple and useful for short texts or keyword sets (e.g. tags, duplicate detection), but for comparing paragraphs TF-IDF + cosine is the more reliable measure.
# - Note: with only two documents, every shared word gets the minimum IDF, so the cosine value is pulled down further. On a larger corpus the TF-IDF weighting becomes even more meaningful.

In [ ]:
# Q5. Sentiment analysis of reviews
# My review is for a food delivery app; a few more reviews are added so that all three classes can be seen.

from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

reviews = [
    "I ordered dinner through the app last night and the food arrived hot and on time. The delivery partner was polite and the tracking was accurate. Absolutely loved the experience!",
    "The app keeps crashing at checkout and my order was delivered forty minutes late. The food was cold and customer support was useless.",
    "The order was delivered at 8 pm. The app shows the restaurant menu and the price of each item.",
    "Great discounts on weekends and the interface is clean and easy to use. Highly recommend it to students.",
    "Fast delivery, fresh food and friendly support. Best food app I have used so far.",
]

In [ ]:
# 1. Polarity & subjectivity (TextBlob) and compound score (VADER)

sia = SentimentIntensityAnalyzer()

rows = []
for i, r in enumerate(reviews, 1):
    blob = TextBlob(r)
    vader = sia.polarity_scores(r)
    rows.append({
        'Review': f'R{i}',
        'TB Polarity': round(blob.sentiment.polarity, 3),
        'TB Subjectivity': round(blob.sentiment.subjectivity, 3),
        'VADER Compound': vader['compound'],
    })
sent_df = pd.DataFrame(rows)
sent_df

In [ ]:
# 2. Classify into Positive / Negative / Neutral
# Using the standard VADER thresholds: compound ≥ 0.05 → Positive, ≤ −0.05 → Negative, otherwise Neutral.

def classify(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    return 'Neutral'

sent_df['Sentiment'] = sent_df['VADER Compound'].apply(classify)
sent_df['Text'] = reviews
pd.set_option('display.max_colwidth', 80)
sent_df[['Review', 'TB Polarity', 'TB Subjectivity', 'VADER Compound', 'Sentiment', 'Text']]

In [ ]:
# - Polarity ranges from −1 (very negative) to +1 (very positive).
# - Subjectivity ranges from 0 (factual) to 1 (opinion). The neutral review R3 is purely factual, so its subjectivity is ~0, while R1 and R5 are strongly opinionated.

In [ ]:
# 3. Word cloud of all positive reviews

from wordcloud import WordCloud, STOPWORDS

positive_text = ' '.join(sent_df.loc[sent_df['Sentiment'] == 'Positive', 'Text'])

wc = WordCloud(width=800, height=400, background_color='white',
               stopwords=STOPWORDS, colormap='viridis').generate(positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud – Positive Reviews')
plt.show()

In [ ]:
# Q6. Text generation with an LSTM
# Training data: my own ~100-word paragraph about the monsoon.

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# Tokenizer lives in keras.preprocessing.text (moved to a legacy module in Keras 3)
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
except ImportError:
    from keras.src.legacy.preprocessing.text import Tokenizer
    from keras.utils import pad_sequences

tf.random.set_seed(42)
np.random.seed(42)

training_text = """The monsoon arrives in north India like a long awaited guest.
Dark clouds gather over the fields and the first drops of rain fall on the dry earth.
The smell of wet soil fills the air and children run outside to play in the rain.
Farmers smile because the rain brings life to their crops.
The rivers rise, the trees turn green and the peacocks dance in the gardens.
People drink hot tea with pakoras and watch the rain from their windows.
The monsoon is not just a season, it is a feeling of hope and joy for every heart in India."""

print(len(training_text.split()), "words")

In [ ]:
# 1. Tokenize with Tokenizer()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text.lower()])
total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)
print(list(tokenizer.word_index.items())[:15])

In [ ]:
# 2. Create n-gram input sequences and build the LSTM model

input_sequences = []
for line in training_text.lower().split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])   # n-gram sequences: [w1,w2], [w1,w2,w3], ...

max_seq_len = max(len(s) for s in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]          # all words except the last
y = input_sequences[:, -1]           # the next word to predict
y = to_categorical(y, num_classes=total_words)

print("Number of sequences:", len(X))
print("X shape:", X.shape, " y shape:", y.shape)

model = Sequential([
    Embedding(input_dim=total_words, output_dim=64),
    LSTM(128),
    Dense(total_words, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_seq_len - 1))
model.summary()

In [ ]:
# 3. Train the model

history = model.fit(X, y, epochs=200, verbose=0)
print(f"Final loss: {history.history['loss'][-1]:.4f}  |  Final accuracy: {history.history['accuracy'][-1]:.4f}")

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1); plt.plot(history.history['loss']); plt.title('Loss'); plt.xlabel('Epoch')
plt.subplot(1, 2, 2); plt.plot(history.history['accuracy'], color='green'); plt.title('Accuracy'); plt.xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# Generate new text from seed words

def generate_text(seed_text, next_words=10, temperature=0.7):
    """Predict one word at a time and append it to the seed."""
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text.lower()])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]

        # temperature sampling: lower = safer, higher = more creative
        probs = np.log(probs + 1e-9) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))
        next_index = np.random.choice(len(probs), p=probs)

        next_word = tokenizer.index_word.get(next_index, '')
        if next_word:
            seed_text += ' ' + next_word
    return seed_text

for seed in ['monsoon', 'farmers', 'people drink']:
    print('•', generate_text(seed, next_words=10))

In [ ]:
# Observation: With only ~100 words of training data the model mostly memorizes the paragraph, so generated lines reuse phrases from it (e.g. *"the rain brings life to their crops"*). Temperature sampling mixes those phrases in new ways; with a lower temperature (e.g. 0.3) the output is closer to the original text, and with a higher one (e.g. 1.2) it becomes more varied but less grammatical. A larger corpus would let the model learn general language patterns instead of memorizing.